# Feature engineering: does it improve the model?

Compares LightGBM on `application_train` with 5-fold stratified CV (ROC-AUC). Only the features change; the model and folds are identical. Feature code is in `src/features.py`; the reasons are in `DECISIONS.md`.

| Variant | What changes |
|---|---|
| A | cleaning only (placeholder, `XNA`): the baseline |
| B | A + engineered features |
| C | B, dropping redundant raw columns |
| D | C + rare categories merged into "Other" |

The first code cell below loads the raw table and separates the label (`TARGET`) from the features.

In [1]:
# Setup: imports and load the raw application table
import sys
sys.path.append("..")  # so src/ can be imported from the notebooks folder

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_score

from src.features import clean, add_features, merge_rare

df = pd.read_csv("../data/raw/HC_application_train.csv")
y = df["TARGET"]                                      # label to predict
raw = df.drop(columns=["TARGET", "SK_ID_CURR"])       # drop label and ID so neither is used as a feature

We score each feature set with one fixed LightGBM model and the same 5 stratified folds, so any AUC change comes from the features alone. Each variant builds on the one before it, which shows what every step contributes.

In [2]:
# Score four feature sets with the same LightGBM model and the same 5 folds
def cv_auc(X):
    X = X.copy()
    # LightGBM reads text columns natively once they are typed as category
    for c in X.select_dtypes(include=["object", "string"]):
        X[c] = X[c].astype("category")
    model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=42, verbose=-1)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # fixed seed: identical folds for every variant
    scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
    return scores.mean(), scores.std(), X.shape[1]

# Each variant adds one step on top of the previous one (A -> B -> C -> D)
variants = {
    "A  baseline (cleaned)": clean(raw),
    "B  + engineered features": add_features(clean(raw)),
    "C  B without redundant columns": add_features(clean(raw), drop_redundant=True),
    "D  C + rare categories merged": merge_rare(add_features(clean(raw), drop_redundant=True)),
}

results = []
for name, X in variants.items():
    mean, std, n = cv_auc(X)
    results.append({"variant": name, "features": n, "AUC mean": round(mean, 4), "AUC std": round(std, 4)})
    print(results[-1])
pd.DataFrame(results)  # summary table, one row per variant

{'variant': 'A  baseline (cleaned)', 'features': 121, 'AUC mean': np.float64(0.7581), 'AUC std': np.float64(0.004)}


{'variant': 'B  + engineered features', 'features': 133, 'AUC mean': np.float64(0.765), 'AUC std': np.float64(0.0042)}


{'variant': 'C  B without redundant columns', 'features': 131, 'AUC mean': np.float64(0.7644), 'AUC std': np.float64(0.004)}


{'variant': 'D  C + rare categories merged', 'features': 131, 'AUC mean': np.float64(0.7646), 'AUC std': np.float64(0.004)}


,variant,features,AUC mean,AUC std
0,A baseline (cleaned),121,0.7581,0.0040
1,B + engineered features,133,0.7650,0.0042
2,C B without redundant columns,131,0.7644,0.0040
3,D C + rare categories merged,131,0.7646,0.0040


**Insight:** Adding engineered features (A to B) lifts mean AUC from 0.7581 to 0.7650 (+0.0069) with 12 extra columns, which is roughly 1.7 times the fold std of about 0.004, so it is the only step that looks like a real gain. Dropping redundant columns (C, 0.7644) and merging rare categories (D, 0.7646) move the score by 0.0006 and 0.0002, which is noise; they are kept for simplicity, not for accuracy.